# PPG-Glucose: Deep Learning Models (Google Colab)
## Leave-One-Subject-Out (LOSO) Cross-Validation

**Models from research papers + strong DL architectures:**
- **1D-CNN** — From *"Non-invasive blood glucose monitoring using PPG signals with various deep learning models"* paper
- **LSTM** — Captures temporal dependencies in PPG beats
- **Bidirectional LSTM** — Better temporal context (forward + backward)
- **CNN-LSTM** — Hybrid: CNN extracts local features, LSTM captures sequence patterns
- **Temporal CNN (TCN)** — Dilated causal convolutions for long-range dependencies
- **Transformer** — Self-attention on PPG windows (state-of-the-art)
- **MLP on features** — Baseline from papers (uses the 111-feature CSV, not raw sequences)

> Upload `dataset_features.csv` and `dataset_sequences.npz` to your Google Drive
> and update the paths in Cell 2 before running.

In [2]:
# ── Cell 1: Install & Mount Drive ─────────────────────────────────────────
# Run this only on Google Colab

import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from time import time
from matplotlib.colors import LinearSegmentedColormap

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Dark theme
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#3a3d4f',   'axes.labelcolor': '#d0d3e8',
    'xtick.color': '#9a9db5',       'ytick.color': '#9a9db5',
    'text.color': '#d0d3e8',        'grid.color': '#2a2d3f',
    'axes.spines.top': False,       'axes.spines.right': False,
    'savefig.dpi': 180,             'savefig.bbox': 'tight',
})

ACCENT  = '#7c6af7'; ACCENT2 = '#38bdf8'; ACCENT3 = '#fb923c'
ACCENT4 = '#34d399'; ACCENT5 = '#f472b6'
GOOD_CMAP = LinearSegmentedColormap.from_list('good', ['#ef4444','#facc15','#34d399'])

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow: 2.20.0
GPU available: True


In [3]:
# ── Cell 2: Load Data ─────────────────────────────────────────────────────
# Load feature CSV (for MLP baseline)
df = pd.read_csv('dataset_features.csv')
EXCLUDE = ['subject_id','glucose','gender','gender_code',
           'age','weight','height','bmi','fs_measured','sqi']
feat_cols = [c for c in df.columns if c not in EXCLUDE]
df[feat_cols] = df[feat_cols].replace([np.inf, -np.inf], np.nan)
df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median())
X_feat   = df[feat_cols].values.astype(np.float32)
y_feat   = df['glucose'].values.astype(np.float32)
groups_feat = df['subject_id'].values

# Load sequences (for CNN/LSTM)
seq_data   = np.load('dataset_sequences.npz', allow_pickle=True)
X_seq      = seq_data['X'].astype(np.float32)   # (N, 130, 2)
y_seq      = seq_data['y'].astype(np.float32)
groups_seq = seq_data['subject_ids']

SEQ_LEN  = X_seq.shape[1]   # 130 timesteps
N_CHAN   = X_seq.shape[2]   # 2 channels (IR, Red)
N_FEAT   = X_feat.shape[1]  # 102 features

print(f'Sequences   : {X_seq.shape}')
print(f'Features    : {X_feat.shape}')
print(f'Glucose     : {y_seq.min():.0f} – {y_seq.max():.0f} mg/dL')
print(f'Subjects    : {len(np.unique(groups_seq))}')

Sequences   : (2771, 130, 2)
Features    : (2771, 102)
Glucose     : 69 – 218 mg/dL
Subjects    : 58


In [4]:
# ── Cell 3: Evaluation Utilities ─────────────────────────────────────────
import gc
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Suppress excessive TF retracing warnings
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

def compute_metrics(y_true, y_pred):
    mae    = float(mean_absolute_error(y_true, y_pred))
    rmse   = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2     = float(r2_score(y_true, y_pred))
    abs_err = np.abs(y_true - y_pred)
    pct_err = abs_err / (y_true + 1e-9) * 100
    zone_a  = float(np.mean((abs_err < 20) | (pct_err < 20)) * 100)
    mape    = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'ZoneA%': zone_a, 'MAPE%': mape}


def loso_evaluate_dl(model_factory, X, y, groups, model_name,
                      epochs=80, batch_size=32, use_scaler=False):
    """
    LOSO-CV for DL models with GPU memory cleanup between folds.
    Key fix: keras.backend.clear_session() after each fold prevents OOM crash.
    """
    unique_subj = np.unique(groups)
    all_true, all_pred = [], []
    per_subj   = []
    histories  = []
    t0 = time()

    for i, test_subj in enumerate(unique_subj):
        tr_mask = groups != test_subj
        te_mask = groups == test_subj
        X_tr, X_te = X[tr_mask], X[te_mask]
        y_tr, y_te = y[tr_mask], y[te_mask]

        if use_scaler:
            sc = RobustScaler()
            X_tr = sc.fit_transform(X_tr)
            X_te = sc.transform(X_te)

        # Clear GPU memory from previous fold — prevents OOM crash
        keras.backend.clear_session()
        gc.collect()

        model = model_factory()
        cb = [
            callbacks.EarlyStopping(monitor='val_loss', patience=12,
                                     restore_best_weights=True, verbose=0),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=6, verbose=0, min_lr=1e-5),
        ]
        hist = model.fit(
            X_tr, y_tr,
            validation_split=0.12,
            epochs=epochs, batch_size=batch_size,
            callbacks=cb, verbose=0
        )
        histories.append(hist.history)  # save history dict not object

        y_pred = model.predict(X_te, verbose=0).flatten()
        m = compute_metrics(y_te, y_pred)
        m['subject'] = test_subj
        per_subj.append(m)
        all_true.append(y_te)
        all_pred.append(y_pred)

        elapsed = time() - t0
        eta     = elapsed / (i + 1) * (len(unique_subj) - i - 1)
        print(f'  [{i+1:02d}/{len(unique_subj)}] {test_subj}  '
              f'MAE={m["MAE"]:.1f}  RMSE={m["RMSE"]:.1f}  '
              f'R2={m["R2"]:.3f}  Ep={len(hist.history["loss"])}  '
              f'ETA {eta:.0f}s', end='\r')

    print()
    all_true = np.concatenate(all_true)
    all_pred = np.concatenate(all_pred)
    overall  = compute_metrics(all_true, all_pred)
    print(f'  => {model_name}: MAE={overall["MAE"]:.2f}  RMSE={overall["RMSE"]:.2f}  '
          f'R2={overall["R2"]:.4f}  ZoneA={overall["ZoneA%"]:.1f}%  ({time()-t0:.0f}s)')
    return overall, pd.DataFrame(per_subj), all_true, all_pred, histories


print('Utilities ready.')


Utilities ready.


In [5]:
# ── Cell 4: Model Architectures ──────────────────────────────────────────

# ── 4b: 1D-CNN on raw sequences ─────────────────────────────────────────
def build_cnn1d(input_shape=(SEQ_LEN, N_CHAN)):
    """From: 'Non-invasive blood glucose monitoring using PPG with DL models' paper."""
    inp = keras.Input(shape=input_shape)
    x = layers.Conv1D(32, 5, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.35)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='huber', metrics=['mae'])
    return model


# ── 4c: LSTM ─────────────────────────────────────────────────────────────
def build_lstm(input_shape=(SEQ_LEN, N_CHAN)):
    """Stacked LSTM for sequential PPG modelling."""
    inp = keras.Input(shape=input_shape)
    x = layers.LSTM(64, return_sequences=True,
                     kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.Dropout(0.25)(x)
    x = layers.LSTM(32)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='huber', metrics=['mae'])
    return model


# ── 4d: Bidirectional LSTM ───────────────────────────────────────────────
def build_bilstm(input_shape=(SEQ_LEN, N_CHAN)):
    """BiLSTM: reads the PPG window in both directions."""
    inp = keras.Input(shape=input_shape)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inp)
    x = layers.Dropout(0.25)(x)
    x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='huber', metrics=['mae'])
    return model


# ── 4e: CNN-LSTM Hybrid ──────────────────────────────────────────────────
def build_cnn_lstm(input_shape=(SEQ_LEN, N_CHAN)):
    """CNN extracts local pulse morphology; LSTM models temporal patterns."""
    inp = keras.Input(shape=input_shape)
    x = layers.Conv1D(32, 5, padding='same', activation='relu')(inp)
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dense(32, activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='huber', metrics=['mae'])
    return model


# ── 4f: Temporal CNN (TCN-like with dilated convolutions) ────────────────
def build_tcn(input_shape=(SEQ_LEN, N_CHAN)):
    """Dilated causal convolutions — captures multi-scale PPG patterns."""
    inp = keras.Input(shape=input_shape)
    x = inp
    for dilation in [1, 2, 4, 8]:
        residual = x
        x = layers.Conv1D(64, 3, dilation_rate=dilation,
                           padding='causal', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
        # Residual connection (match channels)
        if residual.shape[-1] != 64:
            residual = layers.Conv1D(64, 1)(residual)
        x = layers.Add()([x, residual])
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='huber', metrics=['mae'])
    return model


# ── 4g: Transformer (Multi-head Self-Attention) ──────────────────────────
def transformer_encoder(x, head_size=32, num_heads=4, ff_dim=64, dropout=0.2):
    # Multi-head attention
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size,
                                      dropout=dropout)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x1   = layers.LayerNormalization(epsilon=1e-6)(x + attn)
    # Feed-forward
    ff   = layers.Dense(ff_dim, activation='relu')(x1)
    ff   = layers.Dropout(dropout)(ff)
    ff   = layers.Dense(x1.shape[-1])(ff)
    return layers.LayerNormalization(epsilon=1e-6)(x1 + ff)

def build_transformer(input_shape=(SEQ_LEN, N_CHAN)):
    """Transformer encoder for PPG sequences — state-of-the-art."""
    inp = keras.Input(shape=input_shape)
    # Project to embedding dimension
    x = layers.Dense(64)(inp)
    # Positional encoding (simple learnable)
    pos_emb = layers.Embedding(input_dim=input_shape[0], output_dim=64)
    positions = tf.range(start=0, limit=input_shape[0], delta=1)
    x = x + pos_emb(positions)
    # Transformer blocks
    for _ in range(3):
        x = transformer_encoder(x, head_size=32, num_heads=4, ff_dim=128, dropout=0.2)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(5e-4),
                  loss='huber', metrics=['mae'])
    return model


# Print model summaries
for name, factory, shape in [
    ('1D-CNN',         build_cnn1d,     (SEQ_LEN, N_CHAN)),
    ('CNN-LSTM',       build_cnn_lstm,  (SEQ_LEN, N_CHAN)),
    ('Transformer',    build_transformer,(SEQ_LEN, N_CHAN)),
]:
    m = factory(shape)
    print(f'{name}: {m.count_params():,} parameters')

1D-CNN: 40,481 parameters
CNN-LSTM: 41,697 parameters
Transformer: 154,561 parameters


In [6]:
# ── Cell 4: Model Architectures ──────────────────────────────────────────

# ── 4f: Temporal CNN (TCN-like with dilated convolutions) ────────────────
def build_tcn(input_shape=(SEQ_LEN, N_CHAN)):
    """Dilated causal convolutions — captures multi-scale PPG patterns."""
    inp = keras.Input(shape=input_shape)
    x = inp
    for dilation in [1, 2, 4, 8]:
        residual = x
        x = layers.Conv1D(64, 3, dilation_rate=dilation,
                           padding='causal', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
        # Residual connection (match channels)
        if residual.shape[-1] != 64:
            residual = layers.Conv1D(64, 1)(residual)
        x = layers.Add()([x, residual])
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='huber', metrics=['mae'])
    return model


# ── 4g: Transformer (Multi-head Self-Attention) ──────────────────────────
def transformer_encoder(x, head_size=32, num_heads=4, ff_dim=64, dropout=0.2):
    # Multi-head attention
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size,
                                      dropout=dropout)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x1   = layers.LayerNormalization(epsilon=1e-6)(x + attn)
    # Feed-forward
    ff   = layers.Dense(ff_dim, activation='relu')(x1)
    ff   = layers.Dropout(dropout)(ff)
    ff   = layers.Dense(x1.shape[-1])(ff)
    return layers.LayerNormalization(epsilon=1e-6)(x1 + ff)

def build_transformer(input_shape=(SEQ_LEN, N_CHAN)):
    """Transformer encoder for PPG sequences — state-of-the-art."""
    inp = keras.Input(shape=input_shape)
    # Project to embedding dimension
    x = layers.Dense(64)(inp)
    # Positional encoding (simple learnable)
    pos_emb = layers.Embedding(input_dim=input_shape[0], output_dim=64)
    positions = tf.range(start=0, limit=input_shape[0], delta=1)
    x = x + pos_emb(positions)
    # Transformer blocks
    for _ in range(3):
        x = transformer_encoder(x, head_size=32, num_heads=4, ff_dim=128, dropout=0.2)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(5e-4),
                  loss='huber', metrics=['mae'])
    return model


# Print model summaries
for name, factory, shape in [
    ('Transformer',    build_transformer,(SEQ_LEN, N_CHAN)),
]:
    m = factory(shape)
    print(f'{name}: {m.count_params():,} parameters')

Transformer: 154,561 parameters


In [7]:
# ── Cell 5: Run DL Models (LOSO-CV) ──────────────────────────────────────
# MLP-Features removed: ran already, MAE=37.8 (poor cross-subject generalization)
# Models run on raw PPG sequences (better for capturing waveform morphology)
# batch_size=32 reduces peak GPU memory vs 64

all_results   = {}
all_per_subj  = {}
all_preds_map = {}
all_histories = {}

dl_configs = [
    # (name,        factory,           X,      y,      groups,     use_scaler)
    ('TCN',         build_tcn,        X_seq,  y_seq,  groups_seq, False),
    ('Transformer', build_transformer,X_seq,  y_seq,  groups_seq, False),
]

for name, factory, X, y, groups, use_scaler in dl_configs:
    print(f'\n[{name}]')
    fac = factory  # capture current factory in local var to avoid closure bug
    f = (lambda fac=fac: fac((SEQ_LEN, N_CHAN)))

    overall, per_subj, yt, yp, hist = loso_evaluate_dl(
        f, X, y, groups, name,
        epochs=80, batch_size=32, use_scaler=use_scaler
    )
    all_results[name]   = overall
    all_per_subj[name]  = per_subj
    all_preds_map[name] = (yt, yp)
    all_histories[name] = hist
    print(f'  [Saved] {name}')



[TCN]


KeyboardInterrupt: 

In [ ]:
# ── Cell 6: Results Table ─────────────────────────────────────────────────
results_df = pd.DataFrame(all_results).T.round(3)
results_df.index.name = 'Model'
results_df = results_df.sort_values('RMSE')
results_df.to_csv('dl_loso_results.csv')

display(results_df.style
    .background_gradient(subset=['MAE','RMSE'], cmap='RdYlGn_r')
    .background_gradient(subset=['R2','ZoneA%'], cmap='RdYlGn')
    .format({'MAE':'{:.2f}','RMSE':'{:.2f}','R2':'{:.4f}',
             'ZoneA%':'{:.1f}','MAPE%':'{:.2f}'})
)

In [ ]:
# ── Cell 7: Model Comparison Bar Chart ───────────────────────────────────
model_names = list(results_df.index)
n_models    = len(model_names)
palette     = plt.cm.plasma(np.linspace(0.1, 0.9, n_models))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.patch.set_facecolor('#0f1117')

for ax, (metric, title, higher) in zip(axes, [
    ('MAE',    'MAE (mg/dL) — lower is better',    False),
    ('RMSE',   'RMSE (mg/dL) — lower is better',   False),
    ('ZoneA%', 'Clarke Zone A% — higher is better', True),
]):
    ax.set_facecolor('#1a1d27')
    vals = results_df[metric].values
    bars = ax.bar(range(n_models), vals, color=palette, alpha=0.85,
                  edgecolor='#0f1117', linewidth=0.5, width=0.7)
    ax.set_xticks(range(n_models))
    ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.grid(axis='y', alpha=0.35)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{v:.2f}', ha='center', fontsize=8, color='white')
    best_idx = np.argmax(vals) if higher else np.argmin(vals)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)

plt.suptitle('Deep Learning — LOSO Cross-Validation Results',
             fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('dl_comparison.png')
plt.show()

In [ ]:
# ── Cell 8: Actual vs Predicted (Top 4 Models) ───────────────────────────
sorted_models = list(results_df.index)
top4 = sorted_models[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.patch.set_facecolor('#0f1117')
axes = axes.flatten()

for idx, mname in enumerate(top4):
    ax = axes[idx]
    ax.set_facecolor('#1a1d27')
    yt, yp = all_preds_map[mname]
    m = all_results[mname]
    sc = ax.scatter(yt, yp, c=np.abs(yt-yp),
                    cmap='RdYlGn_r', vmin=0, vmax=40,
                    s=16, alpha=0.55, edgecolors='none')
    lims = [min(yt.min(),yp.min())-5, max(yt.max(),yp.max())+5]
    ax.plot(lims, lims,                color='white',   lw=1.5, ls='-',  alpha=0.8)
    ax.plot(lims, [l+20 for l in lims],color='#facc15', lw=1.0, ls='--', alpha=0.6)
    ax.plot(lims, [l-20 for l in lims],color='#facc15', lw=1.0, ls='--', alpha=0.6)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Actual Glucose (mg/dL)')
    ax.set_ylabel('Predicted Glucose (mg/dL)')
    ax.set_title(f'{mname}\nMAE={m["MAE"]:.1f}  RMSE={m["RMSE"]:.1f}  R²={m["R2"]:.3f}  ZoneA={m["ZoneA%"]:.0f}%')
    ax.grid(alpha=0.2)
    plt.colorbar(sc, ax=ax, shrink=0.75, label='|Error| mg/dL')

plt.suptitle('Actual vs Predicted — Top 4 DL Models (LOSO-CV)',
             fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('dl_actual_vs_predicted.png')
plt.show()

In [ ]:
# ── Cell 9: Training Curves (Sample Subject, Best Model) ─────────────────
best_model = sorted_models[0]
sample_hist = all_histories[best_model][0]   # First fold history

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f1117')

for ax, metric, title in zip(axes,
    ['loss', 'mae'],
    ['Training vs Validation Loss (Huber)', 'Training vs Validation MAE']):
    ax.set_facecolor('#1a1d27')
    ep = range(1, len(sample_hist.history['loss']) + 1)
    ax.plot(ep, sample_hist.history[metric],     color=ACCENT2, lw=2, label='Train')
    ax.plot(ep, sample_hist.history[f'val_{metric}'], color=ACCENT5, lw=2, label='Validation')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title.split('(')[-1].replace(')','') if '(' in title else metric.upper())
    ax.set_title(f'{title}\n{best_model} — Fold 1 (S01 as test)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Training History', fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('dl_training_curves.png')
plt.show()

In [ ]:
# ── Cell 10: Error Distribution & Per-Subject Analysis ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0f1117')
palette = plt.cm.plasma(np.linspace(0.1, 0.9, n_models))

ax = axes[0]
ax.set_facecolor('#1a1d27')
for i, mname in enumerate(sorted_models):
    yt, yp = all_preds_map[mname]
    ax.hist(yp - yt, bins=40, alpha=0.45, label=mname,
            color=palette[i], edgecolor='none', density=True)
ax.axvline(0, color='white', lw=1.5)
ax.set_xlabel('Prediction Error (mg/dL)')
ax.set_ylabel('Density')
ax.set_title('Error Distributions — DL Models')
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

ax = axes[1]
ax.set_facecolor('#1a1d27')
abs_errors = {m: np.abs(all_preds_map[m][1] - all_preds_map[m][0])
              for m in sorted_models}
bp = ax.boxplot([abs_errors[m] for m in sorted_models],
                patch_artist=True, medianprops=dict(color='white', lw=2), widths=0.55)
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for el in ['whiskers','caps']:
    for item in bp[el]: item.set_color('#9a9db5')
ax.set_xticks(range(1, n_models+1))
ax.set_xticklabels(sorted_models, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('|Error| (mg/dL)')
ax.set_title('Absolute Error Boxplots — DL Models')
ax.axhline(20, color='#facc15', ls='--', lw=1.3, alpha=0.7, label='20 mg/dL')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.suptitle('DL Error Analysis', fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('dl_error_distributions.png')
plt.show()

print(f'\nAll DL results saved to: dl_results/')
print(f'Best DL model: {sorted_models[0]}  RMSE={all_results[sorted_models[0]]["RMSE"]:.2f} mg/dL')